# Bronze Layer: Fact Tables Ingestion - Order Items

## Executive Summary

This notebook ingests **transactional fact data** into the Bronze layer—the raw business events that drive revenue. While dimension tables describe entities (products, customers), **fact tables capture what actually happened** (orders placed, items purchased, revenue generated).

**Key Outcome:** Immutable historical archive of all e-commerce transactions, enabling revenue analysis, customer behavior tracking, and predictive analytics.

---

## Business Context: Facts vs. Dimensions

### Understanding Fact Tables

**Dimensions = Nouns** (who, what, when, where)  
- Customers, Products, Dates, Brands
- Relatively stable, change infrequently
- Small to medium size (thousands to millions of rows)

**Facts = Verbs** (business events and measurements)  
- Orders placed, items sold, payments received, shipments delivered
- High velocity, constantly growing
- Large volume (millions to billions of rows)
- Contain **metrics** (quantity, price, discount, tax) and **foreign keys** to dimensions

### Why Facts Matter

**For Revenue Analysis:**
- Every order item represents actual revenue
- Aggregating facts produces KPIs: daily sales, average order value, conversion rates

**For Customer Intelligence:**
- Purchase patterns reveal behavior: frequency, recency, basket size
- Enables customer lifetime value (CLV) calculation and churn prediction

**For Product Strategy:**
- Item-level data shows which products drive revenue
- Discount effectiveness, pricing optimization, inventory planning

---

## What This Notebook Accomplishes

### Order Items Fact Table (`brz_order_items`)

**Business Definition:**  
Every row represents **one item within a customer order**—the atomic unit of e-commerce transactions.

**Key Attributes:**
- **Temporal:** `dt` (date), `order_ts` (order timestamp)
- **Identifiers:** `order_id` (order), `customer_id` (who), `product_id` (what), `item_seq` (line number)
- **Metrics:** `quantity`, `unit_price`, `discount_pct`, `tax_amount`
- **Context:** `channel` (web, mobile, store), `coupon_code`
- **Currency:** `unit_price_currency` (multi-currency support)

**Schema Design Note:**  
All fields stored as **strings** in Bronze to handle data quality issues (invalid formats, nulls, special characters). Type casting happens in Silver layer.

---

## Business Value & Use Cases

### Revenue Analytics
- **Daily/Monthly Sales Trends:** Sum of (quantity × unit_price × (1 - discount_pct))
- **Channel Performance:** Compare revenue across web, mobile, store
- **Discount Impact:** Analyze coupon effectiveness and margin erosion

### Customer Behavior
- **Purchase Frequency:** Count distinct order_ids per customer
- **Basket Analysis:** Average items per order, cross-sell opportunities
- **Customer Segmentation:** RFM analysis (Recency, Frequency, Monetary value)

### Product Performance
- **Best Sellers:** Top products by quantity and revenue
- **Price Elasticity:** Relationship between unit_price and quantity sold
- **Inventory Planning:** Demand forecasting based on historical sales

### Operational Intelligence
- **Order Fulfillment:** Track order volume by date for staffing
- **Tax Compliance:** Aggregate tax_amount for regulatory reporting
- **Fraud Detection:** Anomalous discount patterns or channel misuse

---

## Bronze Layer Philosophy: Preserve Raw Truth

### Why Bronze Matters for Facts

🛡️ **Immutable Audit Trail**  
- Every transaction preserved exactly as received from source systems
- Regulatory compliance (SOX, PCI-DSS) requires transaction history
- Dispute resolution: "Show me exactly what the customer was charged"

🔄 **Reprocessing Foundation**  
- Business logic changes (new discount rules, tax calculations)
- Recalculate metrics from raw data without asking IT to re-extract

📊 **Historical Analysis**  
- Year-over-year comparisons require complete history
- Trend detection and seasonality analysis

### Data Quality Approach

**Accept All, Validate Later:**  
- Bronze ingests data "as-is" (all strings, no validation)
- Data quality issues (negative prices, missing IDs) preserved for root cause analysis
- Cleansing and validation happen in Silver layer

**Metadata Enrichment:**  
- `file_name`: Tracks which source file each row came from (lineage)
- `ingest_timestamp`: When data entered the platform (audit trail)

---

## Technical Approach

### Ingestion Workflow

**Step 1: Schema Definition (Cell 4)**  
Explicit schema with all fields as strings
- Prevents schema inference errors
- Handles messy source data gracefully

**Step 2: Load from Unity Catalog Volumes (Cell 5)**  
```python
raw_data_path = "/Volumes/ecommerce/source_data/raw/order_items/landing/*.csv"
```
- Reads all CSV files in landing directory
- Supports incremental file drops (new files automatically picked up)

**Step 3: Metadata Enrichment (Cell 5)**  
- `file_name`: Extracted from Spark metadata for lineage
- `ingest_timestamp`: Current timestamp for audit trail

**Step 4: Write to Delta Table (Cell 7)**  
- Mode: `overwrite` (full refresh for Bronze)
- Format: Delta Lake (ACID transactions, time travel)
- Schema evolution enabled (handles new columns gracefully)

---

## Downstream Impact & Next Steps

Once Bronze fact table is established:

➡️ **Silver Layer Processing** - Data type conversions, metric calculations, deduplication, validation  
➡️ **Gold Layer Aggregations** - Daily sales rollups, customer purchase summaries, product performance metrics  
➡️ **Dimension Enrichment** - Join with customer, product, date dimensions for context  
➡️ **BI Dashboards** - Real-time sales dashboards, executive KPIs, operational reports  
➡️ **ML Pipelines** - Customer churn prediction, demand forecasting, recommendation engines

---

## Key Differences: Facts vs. Dimensions

| **Aspect** | **Fact Tables** | **Dimension Tables** |
|-----------|----------------|---------------------|
| **Represents** | Business events | Business entities |
| **Size** | Millions-billions of rows | Thousands-millions of rows |
| **Growth** | Constantly appending | Relatively stable |
| **Content** | Metrics + foreign keys | Descriptive attributes |
| **Updates** | Rarely updated (append-only) | Updated when entities change |
| **Example** | Order placed, payment made | Customer profile, product catalog |

---

## Execution Prerequisites

✅ Unity Catalog `ecommerce` exists  
✅ Schema `ecommerce.bronze` exists  
✅ Source CSV files in `/Volumes/ecommerce/source_data/raw/order_items/landing/`  
✅ User has `CREATE TABLE` permissions on `ecommerce.bronze` schema  
✅ Serverless compute available (auto-attached on execution)

---

## Notebook Execution Guide

1. **Cell 2:** Import PySpark libraries (StructType, functions)
2. **Cell 3:** Set catalog name (`ecommerce`)
3. **Cell 4:** Define order_items schema (all strings)
4. **Cell 5:** Load CSV files with metadata enrichment
5. **Cell 6:** Preview first 5 rows (validation)
6. **Cell 7:** Write to `ecommerce.bronze.brz_order_items`

**Validation:** After execution, verify row count and schema correctness. Check for unexpected nulls or formatting issues in preview.

---

## Business Stakeholder Impact

| **Stakeholder** | **Enabled Capabilities** |
|----------------|-------------------------|
| **Executive Leadership** | Revenue trends, sales performance KPIs, strategic decision-making |
| **Finance** | Revenue recognition, tax reporting, financial reconciliation |
| **Marketing** | Campaign ROI, coupon effectiveness, customer acquisition cost |
| **Product Management** | Product performance analysis, pricing optimization, portfolio decisions |
| **Sales** | Channel performance, territory analysis, sales rep scorecards |
| **Operations** | Order volume forecasting, fulfillment planning, capacity management |
| **Data Science** | Customer behavior models, demand forecasting, recommendation engines |
| **Compliance/Audit** | Transaction history, regulatory reporting, dispute resolution |

---

**Ready to Execute:** Run cells sequentially to populate the Bronze order_items fact table. This table forms the foundation for all revenue and transactional analytics.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, BooleanType
import pyspark.sql.functions as F

In [0]:
catalog_name = 'ecommerce'

In [0]:
order_items_schema = StructType([
    StructField("dt",                 StringType(), True),
    StructField("order_ts",           StringType(), True),
    StructField("customer_id",        StringType(), True),
    StructField("order_id",           StringType(), True),
    StructField("item_seq",           StringType(), True),
    StructField("product_id",         StringType(), True),
    StructField("quantity",           StringType(), True),
    StructField("unit_price_currency",StringType(), True),
    StructField("unit_price",         StringType(), True),
    StructField("discount_pct",       StringType(), True),
    StructField("tax_amount",         StringType(), True),
    StructField("channel",            StringType(), True),
    StructField("coupon_code",        StringType(), True),
])

In [0]:
# Load data using the schema defined
raw_data_path = "/Volumes/ecommerce/source_data/raw/order_items/landing/*.csv"

df = spark.read.option("header", "true").option("delimiter", ",").schema(order_items_schema).csv(raw_data_path) \
    .withColumn("file_name", F.col("_metadata.file_path")) \
    .withColumn("ingest_timestamp", F.current_timestamp())

In [0]:
display(df.limit(5))

In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.bronze.brz_order_items")